# Novelty Search (Skeleton)
For use in the experiments of the Amorphous Fortress Narrative generation

### General Pseudocode
1. Create initial population of genomes
2. Evaluate each for fitness
3. Evaluate each for novelty against archive genomes
4. Add fit and novel genomes in archive
5. Select new parents of population from novelty archive
6. Mutate children and create new population
7. (Add random back in)
8. Repeat 2-7 for n generations

(Reference: [SimSim](https://github.com/lsoros/simsim/blob/master/simsim.cpp) and [Algorithm Definition](https://algorithmafternoon.com/novelty/novelty_search_algorithm/))

### Setup

In [1]:
# imports
import random
import numpy as np
import json
import spacy
import re
from datetime import datetime
import pyinflect
from sentence_transformers import SentenceTransformer

/Users/mcharit2/Desktop/Research/AF-Narrate/amorphous-fortress-narratives/af-narrate/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# set models for NLP tasks
nlp = spacy.load("en_core_web_sm")
st_model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# import data
ALL_SUBJS = np.load('../bank_files/SUBJECTS.npy', allow_pickle=True)
ALL_OBJS = np.load('../bank_files/OBJECTS.npy', allow_pickle=True)
ALL_VERBS = np.load('../bank_files/VERBS.npy', allow_pickle=True) 
CN_GRAPH = json.load(open('../bank_files/full_word_graph_noweight.json'))

# convert to normal lists
ALL_SUBJS = [str(s) for s in ALL_SUBJS]
ALL_OBJS = [str(o) for o in ALL_OBJS]
ALL_VERBS = [str(v) for v in ALL_VERBS]

print(len(ALL_SUBJS), len(ALL_OBJS), len(ALL_VERBS))

print(random.choices(ALL_SUBJS, k=5), random.choices(ALL_OBJS, k=5), random.choices(ALL_VERBS, k=5))

# TODO: encode all subjects, objects, verbs using sentence transformer model for faster eval and lookup

6018 4849 1506
['skyscraper', 'golfball', 'lightbulb', 'squash', 'government'] ['software', 'flooring', 'stead', 'circus', 'drinking water'] ['glassed', 'attended', 'decided', 'wished', 'did']


In [4]:
# convert all verbs to past tense
# def to_past_tense(verbs):
#     return [verb._.inflect("VBD") if verb._.inflect("VBD") is not None else verb.text for verb in nlp(' '.join(verbs))]

# ALL_VERBS = to_past_tense(ALL_VERBS)

In [5]:
# create verbs and verb encodings
AF_VERBS = ["moved", "died", "cloned", "took", "pushed", "added", "transformed", "blocked", "chased"]
all_af_verb_encs = st_model.encode(AF_VERBS)
af_verb_enc_dict = {AF_VERBS[i]: all_af_verb_encs[i] for i in range(len(AF_VERBS))}

In [6]:
# constants
MC_MUTATE_PERC = 0.25
ENT_MUTATE_PERC = 0.1
VERB_MUTATE_PERC = 0.25
POP_SIZE = 10
NUM_GENERATIONS = 20

In [7]:
# maps a log for usage in the novelty search
class AF_Story:
    def __init__(self, log_file):
        self.log_file = log_file
        with open(log_file, 'r') as f:
            self.og_text = [line.strip() for line in f.readlines()]
        self.ent_ids = self.find_spec_ents()        # dict of entities with subject/object designation
        self.ent_reps = self.get_ent_reps()

        self.ent_order, self.mc_ent = self.find_ents()      # list of all entities in the original text
        self.verb_set = self.find_verbs()          # list of tuples of (subject entity, verb)
        self.verb_order = [v[1] for v in self.verb_set.values()]    # list of all verbs in the original text


    def find_ents(self):
        ''' Find all AF entities in the original text. '''
        af_ents = []
        for line in self.og_text:
            match = re.findall(r'(\[.\..{4}\])', line)
            if match:
                af_ents.extend(match)

        # get highest occuring entity as main character
        random.shuffle(af_ents) # shuffle to avoid biasing first entity as MC
        mc_ent = max(set(af_ents), key = af_ents.count)
        return list(set(af_ents)), mc_ent

    def find_spec_ents(self):
        ''' Identifies entities and whether they are the subject or object in the sentence. '''
        af_ents = {}
        for line in self.og_text:
            match = re.findall(r'(\[.\..{4}\])', line)
            if match:
                for i in range(len(match)):
                    if i == 0:
                        af_ents[match[i]] = 'subject'
                    elif match[i] not in af_ents:
                        af_ents[match[i]] = 'object'
                    
        return af_ents
    
    def get_ent_reps(self):
        ''' Get the symbol representations of the entities in the story '''
        return list(set([e[1] for e in self.ent_ids.keys()]))
    
    def find_verbs(self):
        ''' Find all verbs in the original text '''
        af_verbs = {}
        for i, line in enumerate(self.og_text):
            subj_ent = re.findall(r'(\[.\..{4}\])', line)
            for verb in AF_VERBS:
                if verb in line:
                    af_verbs[i] = (subj_ent[0], verb)
                    break # only take the first verb found
        return af_verbs
    


In [8]:
# test
stupid_story = AF_Story('../logs/stupid_log.txt')
print("Ent IDs:\t" + str(stupid_story.ent_ids))
print("Ent Reps:\t" + str(stupid_story.ent_reps))
print("MC Ent:\t" + str(stupid_story.mc_ent))
print("Ent Order:\t" + str(stupid_story.ent_order))
print("Verb Set:\t" + str(stupid_story.verb_set))
print("Verb Order:\t" + str(stupid_story.verb_order))

Ent IDs:	{'[%.b29d]': 'subject', '[y.c9ae]': 'subject', '[ .d474]': 'subject', '[ .6adb]': 'object', '[$.b14a]': 'subject', '[*.7b93]': 'subject', '[|.c91d]': 'subject', '[}.460f]': 'subject', '[5.6b26]': 'object', '[S.d487]': 'subject', '[!.fdaf]': 'object', '[Y.3a62]': 'object', '[o.8829]': 'object', '[;.f34e]': 'subject', '[9.547d]': 'subject'}
Ent Reps:	['*', '}', '|', 'o', '9', 'Y', '!', '5', 'S', ';', ' ', '%', 'y', '$']
MC Ent:	[%.b29d]
Ent Order:	['[}.460f]', '[%.b29d]', '[5.6b26]', '[*.7b93]', '[o.8829]', '[ .d474]', '[!.fdaf]', '[9.547d]', '[Y.3a62]', '[$.b14a]', '[y.c9ae]', '[|.c91d]', '[;.f34e]', '[S.d487]', '[ .6adb]']
Verb Set:	{3: ('[%.b29d]', 'transformed'), 4: ('[ .d474]', 'cloned'), 5: ('[$.b14a]', 'moved'), 6: ('[*.7b93]', 'moved'), 7: ('[ .d474]', 'transformed'), 8: ('[}.460f]', 'blocked'), 9: ('[S.d487]', 'added'), 10: ('[$.b14a]', 'moved'), 11: ('[|.c91d]', 'pushed'), 12: ('[y.c9ae]', 'chased'), 13: ('[;.f34e]', 'added'), 14: ('[9.547d]', 'pushed')}
Verb Order:	['

In [22]:
# FOR USE WITH THE MUTATION / NOVELTY SEARCH

def get_assoc_dat(mc_ent):
    ''' Gets the associative data (related subjects, objects, and verbs) for a given entity representation '''
    if not mc_ent or mc_ent not in CN_GRAPH:
        return {}
    
    dat = CN_GRAPH[mc_ent]
    verbs = dat.keys()
    assoc_ents = []
    for v in verbs:
        assoc_ents.extend(dat[v])
    subj_ents = [e for e in assoc_ents if e in ALL_SUBJS]
    obj_ents = [e for e in assoc_ents if e in ALL_OBJS]
    return {'subj': subj_ents, 'obj': obj_ents, 'verbs': list(verbs)}


def get_assoc_verbs(ent):
    ''' Gets the associative verbs for a given entity representation '''
    if not ent or ent not in CN_GRAPH:
        return []
    dat = CN_GRAPH[ent]
    verbs = list(dat.keys())
    return verbs

def get_ent_encs(ents):
    ''' Gets the sentence transformer encodings for a list of entities '''
    return {e:st_model.encode(e) for e in ents}


In [66]:
# Object to store the genome info
class FicGenome:
    def __init__(self, story_file:str, mc:str, ent:dict={}, verbs:dict={}):
        '''
            mc:    str
            ent:   {og_story_id: fic_rep_ent}
            verbs: {line: fic_rep_verb} (associated with line number in original story and verb_set)
        '''
        self.story_file = story_file    

        # entities and main character
        self.mc = mc
        self.mc_dat = get_assoc_dat(mc)
        self.ent = ent
        self.ent_encs = get_ent_encs(list(ent.values()))

        # verbs
        self.verbs = verbs

        # algorithm properties
        self.fitness = 0
        self.genome = self.make_genome()


    # ----- MUTATION METHODS ----- #

    def assign_new_ents(self, story, form='random'):
        ''' Generates a new set of entities for the FicGenome object
            form: 'random' | 'assoc'
        '''

        new_ent = {}

        # assume the main character is already set
        mc_subjs = self.mc_dat['subj'][:] if self.mc_dat and form == 'assoc' else []
        mc_objs = self.mc_dat['obj'][:] if self.mc_dat and form == 'assoc' else []
        random.shuffle(mc_subjs)
        random.shuffle(mc_objs)

        # reassign entities based on the associative data
        for ent_id, ent_type in story.ent_ids.items():
            if ent_id == story.mc_ent and self.mc is not None:      # assign the main character
                new_ent[ent_id] = self.mc
                continue
            elif ent_type == 'subject':     # assign subject entity
                if self.mc_dat and len(mc_subjs) > 0:
                    new_ent[ent_id] = mc_subjs.pop()
                else:
                    new_ent[ent_id] = random.choice(ALL_SUBJS)      # out of subject entities or random
            else:             # assign object entity    
                if self.mc_dat and len(mc_objs) > 0:
                    new_ent[ent_id] = mc_objs.pop()
                else:
                    new_ent[ent_id] = random.choice(ALL_OBJS)      # out of object entities
        self.ent = new_ent
        self.ent_encs = get_ent_encs(list(new_ent.values()))

    def assign_new_verbs(self, story, form='random', debug=False):
        ''' Generates a new set of verbs for the FicGenome object
            form: 'random' | 'assoc'
        '''
        if debug:
            print(self.ent)

        new_verbs = {}
        for line, (subj_ent, og_verb) in story.verb_set.items():
            if form == 'random':        # assign random verb
                new_verbs[line] = random.choice(ALL_VERBS)
            elif form == 'assoc':       # assign associated verb to noun
                assoc_verbs = get_assoc_verbs(self.ent[subj_ent]) if subj_ent in self.ent and story.ent_ids[subj_ent] == 'subject' else []
                if len(assoc_verbs) > 0:
                    new_verbs[line] = random.choice(assoc_verbs)
                else:
                    new_verbs[line] = random.choice(ALL_VERBS)
        self.verbs = new_verbs

    def mutate(self, story, mc_form='random', ent_form='random', verb_form='random'):
        ''' Mutates the FicGenome object
            mc:    'random' | 'same'
            ent:   'random' | 'assoc'
            verbs: 'random' | 'assoc'
        '''
        # TODO: mutate based on associations
        self.genome = None # reset genome

        if mc_form == 'random':
            self.mc = random.choice(ALL_SUBJS)
            self.mc_dat = get_assoc_dat(self.mc)

        self.assign_new_ents(story, form=ent_form)      # assign new entities (random or associated)
        self.assign_new_verbs(story, form=verb_form)     # assign new verbs (random or associated)

        # remake the genome based on new values
        self.genome = self.make_genome()


    # ------ NOVELTY / EVOLUTION METHODS ------ #

    def clone(self):
        ''' Returns a separate copy of this object '''
        new_fic = FicGenome(self.story_file, self.mc, {k:v for k,v in self.ent.items()}, {k:v for k,v in self.verbs.items()})
        new_fic.fitness = self.fitness
        new_fic.genome = self.genome
        return new_fic


    def eval(self, af_story, debug=False):
        ''' Evaluates the fitness of the FicGenome object 
            Fitness is based on:
                - Interestingness
                - Semantic closeness of the entities to the main character
                - Semantic closeness of the verb selection to the original verbs
        '''

        
        # TODO: INTERESTINGNESS METRIC
        interestingness_score = random.random()  # placeholder for now (between 0 and 1)
        if debug:
            print(f"- Interestingness Score: {interestingness_score:.4f}")


        # ENT SIMILARITY METRIC
        if len(self.ent_encs) == 0 or af_story.mc_ent not in self.ent:
            ent_score = 0
        else:
            mc_enc = st_model.encode(self.mc)
            cos_sims = []
            for e, enc in self.ent_encs.items():
                if e == self.mc:        # skip the main character
                    continue
                cos_sim = np.dot(mc_enc, enc) / (np.linalg.norm(mc_enc) * np.linalg.norm(enc))
                cos_sims.append(cos_sim)

                if debug:
                    print(f"- MC Ent: {self.mc} | Fic Ent: {e} | Cosine Sim: {float(cos_sim):.4f}")
            ent_score = float(np.mean(cos_sims))    # between 0? and 1


        # VERB SIMILARITY METRIC
        
        # get encodings for verbs
        og_verbs = list(af_story.verb_set.values())
        fic_verbs = list(self.verbs.values())
        og_verb_vecs = [af_verb_enc_dict[v[1]] for v in og_verbs]
        fic_verb_vecs = [st_model.encode(v) for v in fic_verbs]

        # get cosine similarity between verb sets
        cos_sims = []
        for i in range(len(og_verb_vecs)):
            cos_sim = np.dot(og_verb_vecs[i], fic_verb_vecs[i]) / (np.linalg.norm(og_verb_vecs[i]) * np.linalg.norm(fic_verb_vecs[i]))
            cos_sims.append(cos_sim)

            if debug:
                print(f"- OG Verb: {og_verbs[i][1]} | Fic Verb: {fic_verbs[i]} | Cosine Sim: {float(cos_sim):.4f}")

        verb_score = float(np.mean(cos_sims)) # between 0? and 1


        # set the fitness
        self.fitness = (interestingness_score + ent_score + verb_score) / 3.0
        return self.fitness

    def make_genome(self):
        ''' Creates a representation of the genome (ents+verbs) 
            Uses a sentence embedding model from sentence-transformers to convert the genome to a vector
            for comparison with other genomes.
        '''
        all_words = list(self.ent.values()) + list(self.verbs.values())
        all_words = ' '.join(all_words)
        genome = st_model.encode(all_words)
        return genome



    # ------ FILE I/O ------ #

    def generate_story(self, af_story, out_file:str=None):
        ''' Creates a story log based on the genome '''
        new_story = []
        for i, og_line in enumerate(af_story.og_text):
            new_line = og_line
            
            # Replace entities in the original line with their representations
            for ent_id, fic_rep in self.ent.items():
                new_line = new_line.replace(ent_id, f"[{fic_rep}]")

            # Replace verbs in the original line with their representations
            if i in self.verbs:
                af_story_verb = af_story.verb_set[i]
                fic_verb = self.verbs[i]
                new_line = new_line.replace(af_story_verb[1], fic_verb)

            new_story.append(new_line)

        # Write the modified line to the output file
        if out_file is not None:
            with open(out_file, 'w') as f:
                for line in new_story:
                    f.write(line + '\n')

        return new_story
    

    def export_fic(self, af_story, out_file:str=None):
        ''' Exports the data as a JSON file to reimport later '''
        fic_data = {
            'mc': self.mc,
            'ent': self.ent,
            'verbs': self.verbs,
            'fitness': self.fitness,
            'story_file': self.story_file,
            'new_story': '\n'.join(self.generate_story(af_story)) if af_story is not None else None
        }
        if out_file is not None:
            with open(out_file, 'w') as f:
                json.dump(fic_data, f, indent=3)
        return fic_data

    def import_fic(self, dat):
        ''' Imports the data from a JSON file '''
        self.mc = dat['mc']
        self.ent = dat['ent']
        self.verbs = dat['verbs']
        self.fitness = dat['fitness']
        self.story_file = dat['story_file']
        self.genome = self.make_genome()

### Helper Functions

In [75]:
def init_population(size, story, main_char='random', others='random'):
    ''' 
        Initializes a population of FicGenome objects based on the given AF_Story object 
        main_char:    'random' | (specific entity representation)
        others: 'random' | 'assoc' (associated to mc)
    '''
    population = []

    if main_char == 'random' and others == 'random':
        # initialize fully randomly
        for _ in range(size):
            # choose a random MC from the entities in the story
            mc = random.choice(ALL_SUBJS)
            
            # choose random entities from the entities in the story
            ent = {}
            for k,v in story.ent_ids.items():
                if k == story.mc_ent:
                    ent[story.mc_ent] = mc
                else:
                    if v == 'subject':
                        ent[k] = random.choice(ALL_SUBJS)
                    else:
                        ent[k] = random.choice(ALL_OBJS)

            # choose completely random verbs from the verbs in the story
            verbs = {}
            for k,v in story.verb_set.items():
                # pick random verb
                verbs[k] = random.choice(ALL_VERBS)
                

            fic = FicGenome(story.log_file, mc, ent, verbs)
            population.append(fic)

    # make population based on associations to the main character
    elif others == 'assoc':
        for _ in range(size):
            # choose a random MC from the entities in the story
            if main_char != "random" and main_char not in ALL_SUBJS:
                print(f"\t!!!    WARNING    !!!! Main character [{main_char}] not a possible subject in the graph. Defaulting to random choice.")

            mc = random.choice(ALL_SUBJS) if main_char == 'random' else main_char
            fic = FicGenome(story.log_file, mc)

            fic.assign_new_ents(story, form='assoc')      # assign new entities associated with the MC
            fic.assign_new_verbs(story, form='assoc')     # assign new verbs associated

            population.append(fic)


    
    return population

In [76]:
# test generating a story
pop = init_population(1, stupid_story, main_char='random', others='assoc')
fic = pop[0]
print(f"\nFic: MC={fic.mc}, Ents={fic.ent}, Verbs={fic.verbs}")
print(fic.eval(stupid_story, debug=True))


Fic: MC=bomber, Ents={'[%.b29d]': 'bomber', '[y.c9ae]': 'bomb', '[ .d474]': 'membrane', '[ .6adb]': 'bomb', '[$.b14a]': 'highway', '[*.7b93]': 'change', '[|.c91d]': 'greeting', '[}.460f]': 'giggling', '[5.6b26]': 'respect', '[S.d487]': 'typing', '[!.fdaf]': 'breathing', '[Y.3a62]': 'swimming fun', '[o.8829]': 'wedge door', '[;.f34e]': 'fade', '[9.547d]': 'muslim'}, Verbs={3: 'carried', 4: 'separated', 5: 'rode', 6: 'led', 7: 'surrounded', 8: 'impregnated', 9: 'had', 10: 'left', 11: 'emitted', 12: 'made', 13: 'meant', 14: 'pieced'}
- Interestingness Score: 0.3733
- MC Ent: bomber | Fic Ent: bomb | Cosine Sim: 0.6261
- MC Ent: bomber | Fic Ent: membrane | Cosine Sim: 0.1834
- MC Ent: bomber | Fic Ent: highway | Cosine Sim: 0.2379
- MC Ent: bomber | Fic Ent: change | Cosine Sim: 0.2405
- MC Ent: bomber | Fic Ent: greeting | Cosine Sim: 0.2106
- MC Ent: bomber | Fic Ent: giggling | Cosine Sim: 0.2142
- MC Ent: bomber | Fic Ent: respect | Cosine Sim: 0.1610
- MC Ent: bomber | Fic Ent: typi

In [77]:
def is_novel(x, archive, threshold=0.5, debug=False):
    ''' Determines if a FicGenome object is novel compared to an archive of FicGenome objects '''
    if len(archive) == 0:       # nothing in the archive yet, so it's novel!
        return True

    # Get the minimum distance to any genome in the archive
    min_distance = float('inf')
    distances = []
    for a in archive:
        distance = np.linalg.norm(x.genome - a.genome)
        distances.append(distance)
        if distance < min_distance:
            min_distance = distance

    if debug:
        print(f"Distances: {distances}")

    # If the minimum distance is greater than the threshold, the genome is novel
    return min_distance >= threshold


In [78]:
def export_archive(arx, out_file:str, story=None):
    ''' Exports the archive of FicGenome objects to a JSON file '''
    archive_data = [x.export_fic(story) for x in arx]
    with open("../novelty_search_out/archive"+out_file, 'w') as f:
        json.dump(archive_data, f, indent=3)

### Execute Novelty Search

In [79]:
def novelty_search(af_log, fit_threshold=0.5, novel_threshold=0.5, rand_perc=0.2):
    ''' Main novelty search algorithm '''

    # 0. Initialize story representation
    story = AF_Story(af_log)
    
    # 1. Initialize population and archive
    population = init_population(POP_SIZE, story, main_char='random', others='assoc')
    archive = []

    best_fitness = 0
    best_fic = None

    for gen in range(NUM_GENERATIONS):
        print(f"Generation {gen}")

        # 8. Repeat 2-7 for NUM_GENERATIONS
        for indiv in population:

            # 2. Evaluate fitness
            indiv.eval(story)

            # 3+4. Evaluate novelty against archive and add if novel and fit enough
            if is_novel(indiv, archive, novel_threshold) and indiv.fitness > fit_threshold:
                archive.append(indiv.clone())


        # print some stats
        population.sort(key=lambda x: x.fitness, reverse=True)
        fit_scores = [indiv.fitness for indiv in population]
        print(f"  Pop Fitness: max {max(fit_scores):.3f}, min {min(fit_scores):.3f}, avg {sum(fit_scores)/len(fit_scores):.3f}")
        print(f"  Archive size: {len(archive)}")

        if gen % 10 == 0:
            print("   FicGenome of best population individual:")
            print(f"     - Best MC: {population[0].mc}")
            print(f"     - Best Ents: {list(population[0].ent.values())}")
            print(f"     - Best Verbs: {set(population[0].verbs.values())}")

        
        if population[0].fitness > best_fitness:
            best_fitness = population[0].fitness
            best_fic = population[0].clone()
            print(f"  New best fitness: {best_fitness:.3f}")

        # 5. Select new parents from novelty archive
        if len(archive) > 0:
            parents = random.choices(archive, k=int(POP_SIZE*(1-rand_perc)))
        else:
            parents = random.choices(population, k=int(POP_SIZE*(1-rand_perc)))

        # 6. Mutate children from parents
        new_pop = []
        for parent in parents:
            child = parent.clone()
            child.mutate(story, mc_form='random', ent_form='assoc', verb_form='assoc')
            new_pop.append(child)

        # 7. Add random individuals
        rand_amt = (POP_SIZE - len(new_pop))
        for _ in range(int(POP_SIZE*rand_perc)):
            randos = init_population(rand_amt, story, main_char='random', others='assoc')
            new_pop.extend(randos)

        # update population
        population = new_pop

    return archive, best_fic, story

In [80]:
arc, best_fic, story = novelty_search('../logs/stupid_log.txt', fit_threshold=0.5, novel_threshold=0.5, rand_perc=0.2)

Generation 0
  Pop Fitness: max 0.506, min 0.195, avg 0.319
  Archive size: 1
   FicGenome of best population individual:
     - Best MC: seal
     - Best Ents: ['seal', 'rock', 'relative', 'rock', 'renovators', 'diver', 'lodgings', 'shop', 'panpipe', 'grandson', 'institution', 'mineral salts', 'mind control', 'ballons', 'solver']
     - Best Verbs: {'sold', 'sharpened', 'clogged', 'stayed', 'related', 'positioned', 'saved', 'added', 'filled', 'held'}
  New best fitness: 0.506
Generation 1
  Pop Fitness: max 0.486, min 0.239, avg 0.354
  Archive size: 1
Generation 2
  Pop Fitness: max 0.491, min 0.203, avg 0.318
  Archive size: 1
Generation 3
  Pop Fitness: max 0.511, min 0.191, avg 0.364
  Archive size: 2
  New best fitness: 0.511
Generation 4
  Pop Fitness: max 0.490, min 0.177, avg 0.344
  Archive size: 2
Generation 5
  Pop Fitness: max 0.459, min 0.196, avg 0.331
  Archive size: 2
Generation 6
  Pop Fitness: max 0.532, min 0.228, avg 0.362
  Archive size: 2
  New best fitness: 0.53

In [81]:
# save the best fic story
t = datetime.now().strftime("[%m-%d-%Y %H%M]")
print(best_fic.export_fic(story, out_file=f'../novelty_search_out/fic_genomes/best_fic_log-{POP_SIZE}_{NUM_GENERATIONS}_{t}.json'))
best_fic.generate_story(story, out_file=f'../novelty_search_out/gen_stories/best_fic_log-{POP_SIZE}_{NUM_GENERATIONS}_{t}.txt')

{'mc': 'judge', 'ent': {'[%.b29d]': 'judge', '[y.c9ae]': 'fact', '[ .d474]': 'suit', '[ .6adb]': 'murder trial', '[$.b14a]': 'person', '[*.7b93]': 'sentence', '[|.c91d]': 'case', '[}.460f]': 'argument', '[5.6b26]': 'fate', '[S.d487]': 'couple', '[!.fdaf]': 'ticket', '[Y.3a62]': 'person', '[o.8829]': 'law', '[;.f34e]': 'court', '[9.547d]': 'testimony'}, 'verbs': {3: 'allowed', 4: 'wore', 5: 'enjoyed', 6: 'ended', 7: 'wore', 8: 'proved', 9: 'saw', 10: 'called', 11: 'tried', 12: 'considered', 13: 'enforced', 14: 'gave'}, 'fitness': 0.5616363941038522, 'story_file': '../logs/stupid_log.txt', 'new_story': '=====    FORTRESS SEED: [791231]    =====\nFortress initialized! - <0>\n>>> TIME: 2025-08-14 13:21:44 <<<\n<0> [judge] allowed into [fact]\n<1> [suit] wore to [murder trial] at (44, 2)\n<2> [person] enjoyed to (65, 47)\n<3> [sentence] ended to (65, 68)\n<5> [suit] wore into [case]\n<6> [argument] proved by [fate]\n<7> [couple] saw [ticket] at (55, 7)\n<8> [person] called to (50, 78)\n<9> 

['=====    FORTRESS SEED: [791231]    =====',
 'Fortress initialized! - <0>',
 '>>> TIME: 2025-08-14 13:21:44 <<<',
 '<0> [judge] allowed into [fact]',
 '<1> [suit] wore to [murder trial] at (44, 2)',
 '<2> [person] enjoyed to (65, 47)',
 '<3> [sentence] ended to (65, 68)',
 '<5> [suit] wore into [case]',
 '<6> [argument] proved by [fate]',
 '<7> [couple] saw [ticket] at (55, 7)',
 '<8> [person] called to (50, 78)',
 '<9> [case] tried [person]',
 '<10> [fact] considered [law]',
 '<11> [court] enforced [couple] at (37, 88)',
 '<12> [testimony] gave [judge]']

In [82]:
is_novel(best_fic, arc, threshold=1, debug=True)

Distances: [np.float32(0.0), np.float32(1.397351), np.float32(1.3148997), np.float32(1.3238382), np.float32(1.2240672), np.float32(1.3039429)]


np.False_

In [83]:
fic.eval(stupid_story, debug=True)

- Interestingness Score: 0.4025
- MC Ent: bomber | Fic Ent: bomb | Cosine Sim: 0.6261
- MC Ent: bomber | Fic Ent: membrane | Cosine Sim: 0.1834
- MC Ent: bomber | Fic Ent: highway | Cosine Sim: 0.2379
- MC Ent: bomber | Fic Ent: change | Cosine Sim: 0.2405
- MC Ent: bomber | Fic Ent: greeting | Cosine Sim: 0.2106
- MC Ent: bomber | Fic Ent: giggling | Cosine Sim: 0.2142
- MC Ent: bomber | Fic Ent: respect | Cosine Sim: 0.1610
- MC Ent: bomber | Fic Ent: typing | Cosine Sim: 0.2326
- MC Ent: bomber | Fic Ent: breathing | Cosine Sim: 0.2507
- MC Ent: bomber | Fic Ent: swimming fun | Cosine Sim: 0.1894
- MC Ent: bomber | Fic Ent: wedge door | Cosine Sim: 0.1705
- MC Ent: bomber | Fic Ent: fade | Cosine Sim: 0.2550
- MC Ent: bomber | Fic Ent: muslim | Cosine Sim: 0.3577
- OG Verb: transformed | Fic Verb: carried | Cosine Sim: 0.3987
- OG Verb: cloned | Fic Verb: separated | Cosine Sim: 0.2684
- OG Verb: moved | Fic Verb: rode | Cosine Sim: 0.3356
- OG Verb: moved | Fic Verb: led | Cosine S

0.3112173303210893

In [84]:
# # test the fitness evaluation function with the word semantic similarity
# def test_eval(og_verbs, fic_verbs):
#     ''' Evaluates the fitness of the FicGenome object 
#         Fitness is based on:
#             - Interestingness
#             - Semantic closeness of the verb selection to the original verbs
#     '''
    
#     # get encodings for verbs
#     og_verb_vecs = [af_verb_enc_dict[v] for v in og_verbs]
#     all_verbs_vecs = nlp(' '.join(fic_verbs))
#     fic_verb_vecs = [all_verbs_vecs[i] for i in range(len(og_verb_vecs))]

#     # get cosine similarity between verb sets
#     cos_sims = []
#     for i in range(len(og_verb_vecs)):
#         cos_sim = og_verb_vecs[i].similarity(fic_verb_vecs[i])
#         cos_sims.append(cos_sim)

#         print(f"OG Verb: {og_verbs[i]} | Fic Verb: {fic_verbs[i]} | Cosine Sim: {float(cos_sim):.4f}")

#     return float(np.mean(cos_sims))

# test_eval(['moved', 'transformed', 'took'], ['drove', 'altered', 'looked'])

In [85]:
# # create verbs and verb encodings with sentence transformers 
# much more accurate!

# def alt_eval(og_verbs, fic_verbs):
#     ''' Alternative evaluation function using embeddings from sentence transformers '''
#     og_verb_vecs = st_model.encode(og_verbs)
#     fic_verb_vecs = st_model.encode(fic_verbs)
#     cos_sims = []
#     for i in range(len(og_verb_vecs)):
#         cos_sim = np.dot(og_verb_vecs[i], fic_verb_vecs[i]) / (np.linalg.norm(og_verb_vecs[i]) * np.linalg.norm(fic_verb_vecs[i]))
#         cos_sims.append(cos_sim)
#         print(f"OG Verb: {og_verbs[i]} | Fic Verb: {fic_verbs[i]} | Cosine Sim: {float(cos_sim):.4f}")

#     return float(np.mean(cos_sims))

# alt_eval(['moved', 'transformed', 'took'], ['drove', 'transtitioned', 'took'])
